In [ ]:
import pandas as pd 
from statsmodels.stats.multitest import multipletests
pd.options.display.float_format = "{:,.3f}".format

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 150)

INFILE_ENRICH = '/home/grace/work/PPCG_DifferentialGenesetMutation/downstream/results/test/enrichment.tsv'
INFILE_GRPMEANS = '/home/grace/work/PPCG_DifferentialGenesetMutation/downstream/results/test/groupmeans.tsv'

In [ ]:
enrich = pd.read_csv(INFILE_ENRICH, sep='\t', header=0)
enrich['direction'] = (enrich['observed']>enrich['background']).map({True: 'Up', False: 'Down'})
print(enrich['direction'].value_counts())
enrich.head()

In [ ]:
gmeans = pd.read_csv(INFILE_GRPMEANS, sep='\t', header=0)
gmeans.head()

In [ ]:
candidates = list(enrich[enrich['significant']==True]['geneset'].unique())
# candidates = list(enrich[enrich['q_value']<=0.15]['geneset'].unique())

res = pd.read_csv(INFILE_GRPMEANS, sep='\t', header=0)
print(res.shape)
res = res[res['geneset'].isin(candidates)].copy()
print(res.shape)

reject, q_values, _, _ = multipletests(
    res["p_value"], method="fdr_bh", alpha=0.1
)
res["q_value"]     = q_values
res["significant"] = reject

# res = res.sort_values('odds_ratio', ascending=False)
res = res.sort_values('p_value')
res.head()

In [ ]:
res.tail()

In [ ]:
import pandas as pd

# 1. Define a sample dictionary
data = {'a': 100, 'b': 200, 'c': 300, 'd': 400, 'e': 800}

# 2. Convert the dictionary to a pandas Series
my_series = pd.Series(data)

# 3. Print the resulting Series
print(my_series)


In [ ]:
import pandas as pd
import numpy as np


def permute_mutation_matrix(mutation_matrix: pd.DataFrame,
                             gene_sizes: dict[str, float] | None = None,
                             rng: np.random.Generator | None = None) -> pd.DataFrame:
    """
    Return a randomly permuted copy of a binary mutation matrix in which
    each patient's total mutation count (column sum) is exactly preserved.

    For each patient (column), the mutations are redistributed across genes
    by weighted sampling without replacement. Row sums (per-gene mutation
    frequencies) are NOT preserved — genes are sampled freely.

    Parameters
    ----------
    mutation_matrix : pd.DataFrame
        Binary matrix with genes as rows and patients as columns.
        Values must be 0 or 1.
    gene_sizes : dict[str, float], optional
        Mapping of gene name -> size (e.g. coding length in base pairs).
        Sampling probability for each gene is proportional to its size, so
        a gene of size 1000 is sampled 10x more often than one of size 100.
        Genes absent from this dictionary are assigned the median size of
        all provided values. If None, all genes are sampled with equal
        probability (uniform permutation).
    rng : np.random.Generator, optional
        A numpy random Generator for reproducibility, e.g.
        np.random.default_rng(seed=42). If None, a fresh Generator is
        created using entropy from the OS.

    Returns
    -------
    pd.DataFrame
        Permuted binary matrix with the same shape, index, and columns as
        the input. Each column sum equals the corresponding column sum in
        the original matrix.

    Examples
    --------
    >>> rng = np.random.default_rng(42)
    >>> df = pd.DataFrame([[1,0],[0,1],[1,1],[0,0]], columns=["P1","P2"])
    >>> permuted = permute_mutation_matrix(df, rng=rng)
    >>> (permuted.sum(axis=0) == df.sum(axis=0)).all()
    True
    """
    if rng is None:
        rng = np.random.default_rng()

    n_genes  = len(mutation_matrix)
    col_sums = mutation_matrix.sum(axis=0).astype(int)
    permuted = np.zeros((n_genes, len(mutation_matrix.columns)), dtype=np.int8)

    # Build normalised sampling weights aligned to the matrix row order
    if gene_sizes is not None:
        provided_sizes = np.array(list(gene_sizes.values()), dtype=float)
        median_size    = float(np.median(provided_sizes))

        raw_weights = np.array(
            [gene_sizes.get(g, median_size) for g in mutation_matrix.index],
            dtype=float,
        )
        # Normalise to a probability distribution
        sampling_probs = raw_weights / raw_weights.sum()
    else:
        sampling_probs = None   # rng.choice uses uniform by default

    for j, n_mutations in enumerate(col_sums):
        if n_mutations > 0:
            mutated_rows = rng.choice(n_genes, size=n_mutations,
                                      replace=False, p=sampling_probs)
            permuted[mutated_rows, j] = 1

    return pd.DataFrame(
        permuted,
        index   = mutation_matrix.index,
        columns = mutation_matrix.columns,
    )


# =============================================================================
# Example usage
# =============================================================================

if __name__ == "__main__":

    # rng = np.random.default_rng(42)
    rng = np.random.default_rng()

    gene_names = [f"GENE_{i}" for i in range(6)]
    mutation_matrix = pd.DataFrame(
        rng.integers(0, 2, size=(6, 10)),
        index   = gene_names,
        columns = [f"PATIENT_{i}" for i in range(10)],
    )

    # GENE_0 is 10x larger than GENE_1..5
    gene_sizes = {
        "GENE_0": 500,
        "GENE_1": 100,
        "GENE_2": 100,
        "GENE_3": 100,
        "GENE_4": 100,
        # GENE_5 intentionally absent -> assigned median size (100)
    }

    # --- Uniform permutation (no gene_sizes) ---
    permuted_uniform = permute_mutation_matrix(mutation_matrix, rng=rng)

    # --- Weighted permutation ---
    permuted_weighted = permute_mutation_matrix(mutation_matrix,
                                                gene_sizes=gene_sizes, rng=rng)

    print("Column sums preserved (uniform):",
          (permuted_uniform.sum(axis=0) == mutation_matrix.sum(axis=0)).all())
    print("Column sums preserved (weighted):",
          (permuted_weighted.sum(axis=0) == mutation_matrix.sum(axis=0)).all())

    total = mutation_matrix.sum(axis=0).sum()
    print(f"\nMutation share per gene across all patients "
          f"(total mutations = {total}):")
    print(f"{'Gene':<10} {'Uniform %':>10} {'Weighted %':>11} {'Expected %':>11}")
    print("-" * 44)

    # Expected weighted share: GENE_0=1000/(1000+5*100)=~66.7%, others ~6.7%
    provided     = np.array([gene_sizes.get(g, 100) for g in gene_names], dtype=float)
    expected_pct = provided / provided.sum() * 100

    for i, gene in enumerate(gene_names):
        u_pct = permuted_uniform.loc[gene].sum()  / total * 100
        w_pct = permuted_weighted.loc[gene].sum() / total * 100
        print(f"{gene:<10} {u_pct:>10.1f} {w_pct:>11.1f} {expected_pct[i]:>11.1f}")
    
    print()
    print(mutation_matrix)
    print()
    print(permuted_weighted)

In [ ]:
rng = np.random.default_rng()

n_genes = 5
n_patients = 5

gene_names = [f"GENE_{i}" for i in range(n_genes)]
df = pd.DataFrame(
    rng.integers(0, 2, size=(n_genes, n_patients)),
    index   = gene_names,
    columns = [f"PATIENT_{i}" for i in range(n_patients)],
)

df

In [ ]:
df+df

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf


def analyze_gene_associations(df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Test the association between each gene mutation and a continuous response variable,
    adjusting for mutational burden.

    The model fit for each gene is:
        response ~ intercept + mutation_gene + burden

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe where:
            - Column 0 : 'burden'   (int)   total number of mutations per patient
            - Column 1 : 'response' (float) continuous outcome variable
            - Columns 2+: gene columns (0 = no mutation, 1 = mutation)
    alpha : float
        Significance threshold for the 'significant' flag (default 0.05).

    Returns
    -------
    pd.DataFrame
        One row per gene with columns:
            gene         : gene name
            n_mutated    : number of patients with a mutation in this gene
            coef         : regression coefficient for mutation (burden-adjusted effect)
            se           : standard error of the coefficient
            ci_lower     : lower bound of 95% confidence interval
            ci_upper     : upper bound of 95% confidence interval
            t_stat       : t-statistic
            p_value      : two-sided p-value
            p_value_fdr  : Benjamini-Hochberg FDR-adjusted p-value
            significant  : bool — True if p_value_fdr < alpha
            direction    : 'higher', 'lower', or 'n.s.'
        Sorted by p_value ascending.
    """
    burden_col = df.columns[0]
    response_col = df.columns[1]
    gene_cols = df.columns[2:]

    rows = []

    for gene in gene_cols:
        n_mutated = df[gene].sum()

        # Skip genes with no variation (all 0s or all 1s)
        if n_mutated == 0 or n_mutated == len(df):
            rows.append({
                "gene": gene,
                "n_mutated": int(n_mutated),
                "coef": np.nan, "se": np.nan,
                "ci_lower": np.nan, "ci_upper": np.nan,
                "t_stat": np.nan, "p_value": np.nan,
                "p_value_fdr": np.nan, "significant": False,
                "direction": "n.s. (no variance)",
            })
            continue

        model = smf.ols(
            formula=f"response ~ mutation + burden",
            data=df.rename(columns={gene: "mutation", burden_col: "burden", response_col: "response"}),
        ).fit()

        coef = model.params["mutation"]
        se = model.bse["mutation"]
        t_stat = model.tvalues["mutation"]
        p_value = model.pvalues["mutation"]
        ci_lower, ci_upper = model.conf_int().loc["mutation"]

        rows.append({
            "gene": gene,
            "n_mutated": int(n_mutated),
            "coef": round(coef, 4),
            "se": round(se, 4),
            "ci_lower": round(ci_lower, 4),
            "ci_upper": round(ci_upper, 4),
            "t_stat": round(t_stat, 4),
            "p_value": round(p_value, 6),
            "p_value_fdr": np.nan,  # filled below
            "significant": False,
            "direction": "",
        })

    results = pd.DataFrame(rows)

    # --- Benjamini-Hochberg FDR correction on testable genes only ---
    testable = results["p_value"].notna()
    if testable.sum() > 0:
        p_vals = results.loc[testable, "p_value"].values
        fdr = _bh_correction(p_vals)
        results.loc[testable, "p_value_fdr"] = np.round(fdr, 6)
        results.loc[testable, "significant"] = fdr < alpha
        results["direction"] = results.apply(
            lambda r: ("higher" if r["coef"] > 0 else "lower") if r["significant"] else "n.s.",
            axis=1,
        )

    return results.sort_values("p_value").reset_index(drop=True)


def _bh_correction(p_values: np.ndarray) -> np.ndarray:
    """Benjamini-Hochberg FDR correction. Returns adjusted p-values."""
    n = len(p_values)
    order = np.argsort(p_values)
    ranks = np.argsort(order) + 1          # 1-based ranks
    adjusted = p_values * n / ranks
    # Enforce monotonicity (step-down)
    adjusted_ordered = adjusted[order]
    for i in range(n - 2, -1, -1):
        adjusted_ordered[i] = min(adjusted_ordered[i], adjusted_ordered[i + 1])
    fdr = np.empty(n)
    fdr[order] = adjusted_ordered
    return np.minimum(fdr, 1.0)


# ── Example usage ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import warnings
    warnings.filterwarnings("ignore")

    rng = np.random.default_rng(42)

    n_patients = 50
    n_genes = 20
    mut_rate = 0.2

    mutations = rng.binomial(1, mut_rate, size=(n_patients, n_genes))
    burden = mutations.sum(axis=1)

    true_effects = np.zeros(n_genes)
    true_effects[0] = 2.5    # Gene_01: strong positive
    true_effects[1] = -2.0   # Gene_02: strong negative
    true_effects[2] = 1.2    # Gene_03: moderate positive

    response = (
        0.5 * burden
        + (mutations * true_effects).sum(axis=1)
        + rng.normal(0, 2, size=n_patients)
    )

    gene_names = [f"Gene_{i+1:02d}" for i in range(n_genes)]
    df = pd.DataFrame(
        np.column_stack([burden, response, mutations]),
        columns=["burden", "response"] + gene_names,
    ).astype({g: int for g in gene_names})
    df["burden"] = df["burden"].astype(int)

    print()
    print(df.head())
    print()

    results = analyze_gene_associations(df, alpha=0.05)

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 120)
    pd.set_option("display.float_format", "{:.4f}".format)

    print("=" * 80)
    print("Gene–Response Association Analysis (burden-adjusted)")
    print("=" * 80)
    print(results.to_string(index=False))
    print()
    print(f"Significant genes (FDR < 0.05): "
          f"{results.loc[results['significant'], 'gene'].tolist()}")

In [2]:
import pandas as pd

# Example DataFrames with the same index
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]}, index=['row1', 'row2'])
df2 = pd.DataFrame({'C': [5, 6], 'D': [7, 8]}, index=['row1', 'row2'])

print(df1)
print()
print(df2)
print()
# Concatenate column-wise
result_concat = pd.concat([df1, df2], axis=1)
print(result_concat)


      A  B
row1  1  3
row2  2  4

      C  D
row1  5  7
row2  6  8

      A  B  C  D
row1  1  3  5  7
row2  2  4  6  8
